In [3]:
!pip install dash plotly pandas
!pip install dash jupyter-dash
!pip install dash
!pip install dash pandas plotly
!pip install pandas

In [4]:
import dash
from dash import dcc
from dash import html
from dash import Dash, dcc, html

In [5]:
import pandas as pd

df = pd.read_csv("base_datos_ventas_flores_sucias.csv")


df.head()

,Fecha,Año,Tienda,Categoría,Cantidad,Costo_Unitario,Costo_Total,Precio_Venta_Unitario,Ingreso_Total,Ganancia_Neta
0,2024-01-01,2024.0,Portal de las Flores,Girasoles,30.0,12.61,378.30,17.36,520.80,142.50
1,2024-01-01,2024.0,Portal de las Flores,Tulipanes,NaN,25.58,792.98,38.65,1198.15,405.17
2,2024-01-01,2024.0,EcoFlores Norte,Orquídeas,39.0,47.51,1852.89,63.08,2460.12,607.23
3,2024-01-01,2024.0,Jardín Express,Girasoles,39.0,11.00,429.00,17.70,690.30,261.30
4,2024-01-02,2024.0,Florería Central,Girasoles,46.0,16.07,739.22,26.70,1228.20,488.98


In [9]:
import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import plotly.graph_objects as go

# ======================================================
# CARGAR DATASET
# ======================================================

df = pd.read_csv("base_datos_ventas_flores_sucias.csv")

# ======================================================
# LIMPIEZA
# ======================================================

df = df.dropna()

df["Fecha"] = pd.to_datetime(df["Fecha"])

df["Año"] = df["Fecha"].dt.year
df["Mes"] = df["Fecha"].dt.month_name()

# ======================================================
# INICIAR APP
# ======================================================

app = dash.Dash(__name__)
server = app.server

# ======================================================
# ESTILO TARJETAS KPI
# ======================================================

CARD_STYLE = {
    "padding": "20px",
    "borderRadius": "15px",
    "color": "white",
    "width": "23%",
    "textAlign": "center",
    "boxShadow": "2px 2px 10px rgba(0,0,0,0.2)"
}

# ======================================================
# LAYOUT
# ======================================================

app.layout = html.Div([

    # ==================================================
    # TITULO
    # ==================================================

    html.H1(
        "🌸 Dashboard Ejecutivo - Florerías",
        style={
            "textAlign": "center",
            "color": "white",
            "marginBottom": "10px"
        }
    ),

    html.P(
        """
        Análisis interactivo de ventas, ingresos y rentabilidad
        de las diferentes categorías de flores.
        """,
        style={
            "textAlign": "center",
            "color": "white",
            "fontSize": "18px",
            "marginBottom": "30px"
        }
    ),

    # ==================================================
    # FILTROS
    # ==================================================

    html.Div([

        # TIENDAS
        html.Div([

            html.Label(
                "Seleccionar Tienda",
                style={"color": "white"}
            ),

            dcc.Dropdown(
                id="filtro_tienda",

                options=[
                    {"label": i, "value": i}
                    for i in df["Tienda"].unique()
                ],

                value=df["Tienda"].unique().tolist(),

                multi=True
            )

        ], style={"width": "32%"}),

        # CATEGORIAS
        html.Div([

            html.Label(
                "Seleccionar Categoría",
                style={"color": "white"}
            ),

            dcc.Dropdown(
                id="filtro_categoria",

                options=[
                    {"label": i, "value": i}
                    for i in df["Categoría"].unique()
                ],

                value=df["Categoría"].unique().tolist(),

                multi=True
            )

        ], style={"width": "32%"}),

        # AÑOS
        html.Div([

            html.Label(
                "Seleccionar Año",
                style={"color": "white"}
            ),

            dcc.Dropdown(
                id="filtro_anio",

                options=[
                    {"label": i, "value": i}
                    for i in sorted(df["Año"].unique())
                ],

                value=sorted(df["Año"].unique()),

                multi=True
            )

        ], style={"width": "32%"})

    ],
    style={
        "display": "flex",
        "justifyContent": "space-between",
        "marginBottom": "30px"
    }),

    # ==================================================
    # KPIs
    # ==================================================

    html.Div(
        id="kpis",

        style={
            "display": "flex",
            "justifyContent": "space-between",
            "marginBottom": "40px"
        }
    ),

    # ==================================================
    # GRAFICO 1
    # ==================================================

    dcc.Graph(id="linea_ventas"),

    # ==================================================
    # GRAFICOS 2 Y 3
    # ==================================================

    html.Div([

        html.Div([
            dcc.Graph(id="bar_categoria")
        ], style={"width": "49%"}),

        html.Div([
            dcc.Graph(id="bar_tienda")
        ], style={"width": "49%"})

    ],
    style={
        "display": "flex",
        "justifyContent": "space-between"
    }),

    # ==================================================
    # GRAFICOS 4 Y 5
    # ==================================================

    html.Div([

        html.Div([
            dcc.Graph(id="boxplot")
        ], style={"width": "49%"}),

        html.Div([
            dcc.Graph(id="scatter")
        ], style={"width": "49%"})

    ],
    style={
        "display": "flex",
        "justifyContent": "space-between"
    }),

    # ==================================================
    # GRAFICOS 6 Y 7
    # ==================================================

    html.Div([

        html.Div([
            dcc.Graph(id="pie")
        ], style={"width": "49%"}),

        html.Div([
            dcc.Graph(id="heatmap")
        ], style={"width": "49%"})

    ],
    style={
        "display": "flex",
        "justifyContent": "space-between"
    })

],
style={
    "backgroundColor": "#111111",
    "padding": "30px",
    "fontFamily": "Arial"
})

# ======================================================
# CALLBACK
# ======================================================

@app.callback(

    [
        Output("kpis", "children"),
        Output("linea_ventas", "figure"),
        Output("bar_categoria", "figure"),
        Output("bar_tienda", "figure"),
        Output("boxplot", "figure"),
        Output("scatter", "figure"),
        Output("pie", "figure"),
        Output("heatmap", "figure")
    ],

    [
        Input("filtro_tienda", "value"),
        Input("filtro_categoria", "value"),
        Input("filtro_anio", "value")
    ]

)

def actualizar_dashboard(tiendas, categorias, anios):

    # ==================================================
    # FILTRAR DATOS
    # ==================================================

    dff = df[
        (df["Tienda"].isin(tiendas)) &
        (df["Categoría"].isin(categorias)) &
        (df["Año"].isin(anios))
    ]

    # ==================================================
    # KPIs
    # ==================================================

    ingresos = dff["Ingreso_Total"].sum()
    ganancias = dff["Ganancia_Neta"].sum()
    unidades = dff["Cantidad"].sum()
    ticket = dff["Ingreso_Total"].mean()

    kpis = [

        html.Div([

            html.H3("💰 Ingresos"),
            html.H1(f"${ingresos:,.0f}")

        ],
        style={**CARD_STYLE, "backgroundColor": "#16A085"}),

        html.Div([

            html.H3("📈 Ganancia"),
            html.H1(f"${ganancias:,.0f}")

        ],
        style={**CARD_STYLE, "backgroundColor": "#2980B9"}),

        html.Div([

            html.H3("🛒 Unidades"),
            html.H1(f"{unidades:,.0f}")

        ],
        style={**CARD_STYLE, "backgroundColor": "#8E44AD"}),

        html.Div([

            html.H3("🧾 Ticket Promedio"),
            html.H1(f"${ticket:,.0f}")

        ],
        style={**CARD_STYLE, "backgroundColor": "#D35400"})

    ]

    # ==================================================
    # LINEA TEMPORAL
    # ==================================================

    ventas_fecha = dff.groupby(
        "Fecha"
    )["Ingreso_Total"].sum().reset_index()

    fig_linea = px.line(
        ventas_fecha,
        x="Fecha",
        y="Ingreso_Total",
        title="📈 Evolución Temporal de Ventas"
    )

    fig_linea.update_layout(template="plotly_dark")

    # ==================================================
    # BARRAS CATEGORIA
    # ==================================================

    categoria = dff.groupby(
        "Categoría"
    )["Ganancia_Neta"].sum().reset_index()

    fig_categoria = px.bar(
        categoria,
        x="Categoría",
        y="Ganancia_Neta",
        color="Categoría",
        title="🌸 Ganancia por Categoría"
    )

    fig_categoria.update_layout(template="plotly_dark")

    # ==================================================
    # BARRAS TIENDA
    # ==================================================

    tiendas_df = dff.groupby(
        "Tienda"
    )["Ingreso_Total"].sum().reset_index()

    fig_tienda = px.bar(
        tiendas_df,
        x="Tienda",
        y="Ingreso_Total",
        color="Tienda",
        title="🏪 Ingresos por Tienda"
    )

    fig_tienda.update_layout(template="plotly_dark")

    # ==================================================
    # BOXPLOT
    # ==================================================

    fig_box = px.box(
        dff,
        x="Categoría",
        y="Ingreso_Total",
        color="Categoría",
        title="📦 Distribución de Ingresos"
    )

    fig_box.update_layout(template="plotly_dark")

    # ==================================================
    # SCATTER
    # ==================================================

    fig_scatter = px.scatter(
        dff,
        x="Costo_Unitario",
        y="Precio_Venta_Unitario",
        size="Ganancia_Neta",
        color="Categoría",
        hover_data=["Tienda"],
        title="💲 Relación Costos vs Precio"
    )

    fig_scatter.update_layout(template="plotly_dark")

    # ==================================================
    # PIE CHART
    # ==================================================

    pie_df = dff.groupby(
        "Categoría"
    )["Ingreso_Total"].sum().reset_index()

    fig_pie = px.pie(
        pie_df,
        names="Categoría",
        values="Ingreso_Total",
        title="🥧 Participación de Ventas"
    )

    fig_pie.update_layout(template="plotly_dark")

    # ==================================================
    # HEATMAP
    # ==================================================

    columnas = [
        "Cantidad",
        "Costo_Unitario",
        "Costo_Total",
        "Precio_Venta_Unitario",
        "Ingreso_Total",
        "Ganancia_Neta"
    ]

    corr = dff[columnas].corr()

    fig_heatmap = go.Figure(

        data=go.Heatmap(
            z=corr.values,
            x=corr.columns,
            y=corr.columns,
            colorscale="RdBu",
            zmin=-1,
            zmax=1
        )

    )

    fig_heatmap.update_layout(
        title="🔥 Matriz de Correlación",
        template="plotly_dark"
    )

    # ==================================================
    # RETURN
    # ==================================================

    return (
        kpis,
        fig_linea,
        fig_categoria,
        fig_tienda,
        fig_box,
        fig_scatter,
        fig_pie,
        fig_heatmap
    )

# ======================================================
# EJECUTAR
# ======================================================

if __name__ == "__main__":
    app.run(debug=True, port=8051)